# Perform feature_engineering

In [1]:
import pandas as pd
import numpy as np

cleaned_df = pd.read_csv('c:/code/python_jupyter/project_resources/cleaned_survey_results.csv')
print(cleaned_df.describe())

        cf_ab_score     zas_score           bsi
count  29991.000000  29991.000000  29991.000000
mean       0.537350      6.096529      0.305925
std        0.141866      5.517959      0.460806
min        0.250000      0.000000      0.000000
25%        0.500000      0.000000      0.000000
50%        0.500000      6.000000      0.000000
75%        0.670000      9.000000      1.000000
max        0.750000     20.000000      1.000000


In [4]:

# --- Step 1: Categorize Age into Age Groups ---
age_bins = [17, 25, 35, 45, 55, 70, 150]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-70', '70+']
cleaned_df['age_group'] = pd.cut(cleaned_df['age'], bins=age_bins, labels=age_labels)
cleaned_df = cleaned_df.drop(columns=['age'])

# --- Step 2: Create cf_ab_score ---
freq_map = {'0-2 times': 1, '3-4 times': 2, '5-7 times': 3}
awareness_map = {'0 to 1': 1, '2 to 4': 2, 'above 4': 3}

freq_score = cleaned_df['consume_frequency(weekly)'].map(freq_map)
awareness_score = cleaned_df['awareness_of_other_brands'].map(awareness_map)

cleaned_df['cf_ab_score'] = (freq_score / (awareness_score + freq_score)).round(2)

# --- Step 3: Create Zone Affluence Score (ZAS) ---
zone_map = {'Rural': 1, 'Semi-Urban': 2, 'Urban': 3, 'Metro': 4}

def get_income_score(val):
    if pd.isna(val) or val == 'Not Reported':
        return 0
    val_clean = str(val).replace(' ', '')
    if '<10L' in val_clean:
        return 1
    elif '10L-15L' in val_clean:
        return 2
    elif '16L-25L' in val_clean:
        return 3
    elif '26L-35L' in val_clean:
        return 4
    elif '>35L' in val_clean:
        return 5
    return 0

zone_score = cleaned_df['zone'].map(zone_map)
income_score = cleaned_df['income_levels'].apply(get_income_score)
cleaned_df['zas_score'] = zone_score * income_score

# --- Step 4: Brand Switching Indicator (BSI) ---
bsi_condition = (cleaned_df['current_brand'] != 'Established') & (
    cleaned_df['reasons_for_choosing_brands'].isin(['Price', 'Quality'])
)
cleaned_df['bsi'] = bsi_condition.astype(int)

# --- Final Cleaning Step: Removing Logical Outliers ---
# Removing illogical entries (e.g. Students aged 56-70)
logical_outliers = (cleaned_df['occupation'] == 'Student') & (cleaned_df['age_group'] == '56-70')
df_final = cleaned_df[~logical_outliers].copy()
df_final.to_csv('c:/code/python_jupyter/project_resources/engineered_survey_results.csv', index=False)

print("Data feature_engineering complete.")

Data feature_engineering complete.
